# CSI sur grille fine

Ce notebook reprend la logique de `eval_checkpoints_fixed_hourly.ipynb` mais calcule le CSI apres interpolation des predictions et de la verite terrain sur une grille reguliere commune construite a partir du maillage fin `maillage_3.slf`.

Points a verifier avant execution :
- `COARSE_MESH_PATHS`
- `FINE_MESH_PATH`
- les listes `DYNAMIC_DIR` / `HYDRO_DIR`
- `GRID_RESOLUTION` si le cout memoire est trop eleve


In [ ]:
import os
import sys
import math
import pickle
from pathlib import Path

import dgl
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Ajouter le repo au PYTHONPATH
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

from python.create_dgl_dataset import (
    TelemacDataset,
    TelemacDatasetWithQ,
    unpack_dynamic_sample,
)
from python.CustomMeshGraphNet import MeshGraphNet
from python.python_code.data_manip.extraction.telemac_file import TelemacFile
from python.python_code.data_manip.formats.regular_grid import interpolate_on_grid
from modulus.launch.utils import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =====================
# Parametres utilisateur
# =====================

DATA_DIRS = {
    "multimesh": "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Multimesh_8_32.bin",
    "normal": "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Mesh8_base.bin",
}
DEFAULT_MESH = "multimesh"

# Support de calcul du surrogate.
# Adapter si les .slf x8 ne sont pas a cet emplacement.
COARSE_MESH_PATHS = {
    "normal": "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Mesh8_corrige.slf",
    # Dans beaucoup de pipelines, le multimesh garde le meme support nodal que le mesh x8
    # et ajoute seulement des arêtes longues portees. Si ce n'est pas ton cas, remplace ici.
    "multimesh": "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Mesh8_corrige.slf",
}

# Verite terrain sur maillage fin
FINE_MESH_PATH = "/work/m24046/m24046mrcr/dataset_x8_avec_ts/short/maillage_3.slf"

DYNAMIC_DIR = [
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_1_peak_2600_Group_1_peak_2600_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_2_peak_1000_Group_2_peak_1000_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_2_peak_1200_Group_2_peak_1200_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_2_peak_1600_Group_2_peak_1600_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_4_peak_2000_Group_4_peak_2000_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_1_peak_1200_Group_1_peak_1200_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_1_peak_2400_Group_1_peak_2400_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_3_peak_3400_Group_3_peak_3400_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_1_peak_1400_Group_1_peak_1400_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_1_peak_2000_Group_1_peak_2000_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_1_peak_2200_Group_1_peak_2200_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_2_peak_3600_Group_2_peak_3600_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_3_peak_2200_Group_3_peak_2200_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_3_peak_2800_Group_3_peak_2800_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_4_peak_1200_Group_4_peak_1200_0_35-64_interpolated.pkl",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_4_peak_3000_Group_4_peak_3000_0_35-64_interpolated.pkl",
]

USE_Q_FEATURE = True
HYDRO_DIR = [
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_2600.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_2_peak_1000.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_2_peak_1200.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_2_peak_1600.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_4_peak_2000.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_1200.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_2400.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_3_peak_3400.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_1400.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_2000.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_2200.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_2_peak_3600.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_3_peak_2200.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_3_peak_2800.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_4_peak_1200.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_4_peak_3000.liq",
]

METHODS = [
    {
        "name": "Experience 1",
        "ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience1/Seed0/",
        "epoch": 900,
        "use_q_feature": False,
        "mesh": "normal",
    },
    {
        "name": "Experience 2",
        "ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience7/Seed0/",
        "epoch": 900,
        "use_q_feature": True,
        "mesh": "normal",
    },
    {
        "name": "Experience 3",
        "ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience9/Seed0/",
        "epoch": 900,
        "use_q_feature": True,
        "mesh": "normal",
    },
    {
        "name": "Experience 4",
        "ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience2/Seed0/",
        "epoch": 900,
        "use_q_feature": False,
        "mesh": "multimesh",
    },
    {
        "name": "Experience 5",
        "ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience8/Seed0/",
        "epoch": 900,
        "use_q_feature": True,
        "mesh": "multimesh",
    },
    {
        "name": "Experience 6",
        "ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience10/Seed0/",
        "epoch": 900,
        "use_q_feature": True,
        "mesh": "multimesh",
    },
]

DT_SECONDS = 1800.0

NUM_EDGE_FEATURES = 3
NUM_OUTPUT_FEATURES = 3
MP_LAYERS = 10
DO_CONCAT_TRICK = True
NUM_PROCESSOR_CHECKPOINT_SEGMENTS = 0

MAX_HOURS = 12
HOUR_IN_STEPS = max(1, int(round(3600.0 / DT_SECONDS)))
HORIZONS_STEPS = [h * HOUR_IN_STEPS for h in range(1, MAX_HOURS + 1)]
MAX_SEQUENCES = 205
THRESHOLD_M = 0.05
GRID_RESOLUTION = (500, 500)

OUTPUT_CSV_PATH = "/work/m24046/m24046mrcr/paper/eval_csi_fine_grid_hourly.csv"


def derive_fine_dynamic_files(dynamic_files):
    fine_files = []
    for path in dynamic_files:
        fine_path = path.replace("/shortx8/", "/short/").replace("_interpolated.pkl", ".pkl")
        if fine_path == path:
            raise ValueError(
                "Impossible de deriver le fichier fin depuis le chemin x8: "
                f"{path}"
            )
        fine_files.append(fine_path)
    return fine_files


def build_model(num_input_features):
    return MeshGraphNet(
        num_input_features,
        NUM_EDGE_FEATURES,
        NUM_OUTPUT_FEATURES,
        processor_size=MP_LAYERS,
        hidden_dim_processor=64,
        hidden_dim_node_encoder=64,
        hidden_dim_edge_encoder=64,
        hidden_dim_node_decoder=64,
        do_concat_trick=DO_CONCAT_TRICK,
        num_processor_checkpoint_segments=NUM_PROCESSOR_CHECKPOINT_SEGMENTS,
    )


def load_model_checkpoint(model, ckpt_dir, epoch):
    load_checkpoint(ckpt_dir, models=model, device=device, epoch=epoch)
    model.to(device)
    model.eval()
    return model


def build_dataset(
    sequence_length,
    ckpt_dir,
    use_q_feature,
    data_dir,
    overlap=1,
    split="test",
    dynamic_dir=None,
    hydro_dir=None,
):
    dynamic_files = dynamic_dir if dynamic_dir is not None else DYNAMIC_DIR
    hydro_files = hydro_dir if hydro_dir is not None else HYDRO_DIR

    if use_q_feature:
        ds = TelemacDatasetWithQ(
            name=f"eval_{split}",
            data_dir=data_dir,
            dynamic_data_files=dynamic_files,
            hydro_data_files=hydro_files,
            split=split,
            ckpt_path=ckpt_dir,
            normalize=True,
            sequence_length=sequence_length,
            overlap=overlap,
            dt_seconds=DT_SECONDS,
        )
    else:
        ds = TelemacDataset(
            name=f"eval_{split}",
            data_dir=data_dir,
            dynamic_data_files=dynamic_files,
            split=split,
            ckpt_path=ckpt_dir,
            normalize=True,
            sequence_length=sequence_length,
            overlap=overlap,
        )
    return ds


def load_dynamic_sequences(dynamic_files, sequence_length, overlap=1):
    sequences = []
    step = max(1, sequence_length - overlap)
    for file_path in dynamic_files:
        with open(file_path, "rb") as f:
            dynamic_data = pickle.load(f)
        for i in range(0, len(dynamic_data) - sequence_length + 1, step):
            sequences.append(dynamic_data[i:i + sequence_length])
    return sequences


def _denorm(xn, mean, std):
    return xn * std + mean


def _renorm(x, mean, std):
    return (x - mean) / (std + 1e-12)


def csi_from_binary(pred_mask, gt_mask):
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    denom = tp + fp + fn
    return float(tp / denom) if denom > 0 else math.nan


_MESH_CACHE = {}
_GRID_CACHE = {}


def get_mesh(mesh_path):
    if mesh_path not in _MESH_CACHE:
        _MESH_CACHE[mesh_path] = TelemacFile(mesh_path)
    return _MESH_CACHE[mesh_path]


def get_common_grid(fine_mesh_path, grid_resolution):
    key = (fine_mesh_path, tuple(grid_resolution))
    if key not in _GRID_CACHE:
        fine_mesh = get_mesh(fine_mesh_path)
        ones = np.ones(fine_mesh.npoin2, dtype=np.float64)
        mask_field, grid = interpolate_on_grid(
            fine_mesh.tri,
            ones,
            grid_resolution=grid_resolution,
        )
        domain_mask = np.ma.getmaskarray(np.ma.asarray(mask_field))
        _GRID_CACHE[key] = (grid, domain_mask)
    return _GRID_CACHE[key]


def interpolate_depth_on_grid(tri, h_values, grid):
    h_grid, _ = interpolate_on_grid(tri, np.asarray(h_values, dtype=np.float64), grid=grid)
    return np.ma.asarray(h_grid)


def compute_grid_csi(h_pred_nodes, pred_mesh, h_gt_nodes, fine_mesh, grid, threshold, domain_mask):
    if len(h_pred_nodes) != pred_mesh.npoin2:
        raise ValueError(
            f"Prediction node count mismatch: {len(h_pred_nodes)} vs {pred_mesh.npoin2}"
        )
    if len(h_gt_nodes) != fine_mesh.npoin2:
        raise ValueError(
            f"Fine GT node count mismatch: {len(h_gt_nodes)} vs {fine_mesh.npoin2}"
        )

    pred_grid = interpolate_depth_on_grid(pred_mesh.tri, h_pred_nodes, grid)
    gt_grid = interpolate_depth_on_grid(fine_mesh.tri, h_gt_nodes, grid)

    pred_masked = np.ma.asarray(pred_grid)
    gt_masked = np.ma.asarray(gt_grid)

    mask = (
        np.ma.getmaskarray(pred_masked)
        | np.ma.getmaskarray(gt_masked)
        | domain_mask
    )

    pred_vals = np.asarray(pred_masked.filled(np.nan))
    gt_vals = np.asarray(gt_masked.filled(np.nan))
    valid = (~mask) & np.isfinite(pred_vals) & np.isfinite(gt_vals)

    if not np.any(valid):
        return math.nan

    pred_binary = np.zeros_like(valid, dtype=bool)
    gt_binary = np.zeros_like(valid, dtype=bool)
    pred_binary[valid] = pred_vals[valid] >= threshold
    gt_binary[valid] = gt_vals[valid] >= threshold

    return csi_from_binary(pred_binary[valid], gt_binary[valid])


def evaluate_model_fine_grid(
    model,
    ds,
    fine_sequences,
    horizons_steps,
    use_q_feature,
    coarse_mesh,
    fine_mesh,
    grid,
    domain_mask,
    max_sequences=10,
    threshold=0.05,
):
    stats = ds.node_stats
    dyn_start = ds.base_graph.ndata["static"].shape[1]
    dyn_len = 4 if use_q_feature else 3

    mx = torch.tensor(
        [stats["h"].item(), stats["u"].item(), stats["v"].item()],
        device=device,
    )
    sx = torch.tensor(
        [stats["h_std"].item(), stats["u_std"].item(), stats["v_std"].item()],
        device=device,
    )
    dy_mean = torch.tensor(
        [stats["delta_h"].item(), stats["delta_u"].item(), stats["delta_v"].item()],
        device=device,
    )
    dy_std = torch.tensor(
        [
            stats["delta_h_std"].item(),
            stats["delta_u_std"].item(),
            stats["delta_v_std"].item(),
        ],
        device=device,
    )

    agg = {h: {"csi_fine": []} for h in horizons_steps}
    nseq = min(max_sequences, len(ds), len(fine_sequences))
    max_h = max(horizons_steps)

    for idx in range(nseq):
        graphs = ds[idx]
        fine_seq = fine_sequences[idx]
        if len(graphs) <= max_h or len(fine_seq) <= max_h:
            continue

        g = graphs[0].to(device)
        static_part = g.ndata["x"][:, :dyn_start]
        xn_t_full = g.ndata["x"][:, dyn_start:dyn_start + dyn_len]

        onehot = static_part[:, :4]
        q_mask = (onehot == torch.tensor([0, 0, 1, 0], device=device)).all(dim=1)
        h_mask = (onehot == torch.tensor([0, 1, 0, 0], device=device)).all(dim=1)

        for t in range(max_h):
            with torch.no_grad():
                y_pred_n = model(g.ndata["x"], g.edata["x"], g)

            xn_t = xn_t_full[:, :3]
            x_t = _denorm(xn_t, mx, sx)
            y_pred = _denorm(y_pred_n, dy_mean, dy_std)
            x_t1 = x_t + y_pred

            x_gt_full_n = graphs[t + 1].ndata["x"][:, dyn_start:dyn_start + dyn_len].to(device)
            x_gt_n = x_gt_full_n[:, :3]
            x_gt = _denorm(x_gt_n, mx, sx)

            x_t1[q_mask] = x_gt[q_mask]
            x_t1[h_mask, 0:1] = x_gt[h_mask, 0:1]

            step = t + 1
            if step in horizons_steps:
                fine_x_t, fine_y_t, _ = unpack_dynamic_sample(fine_seq[t])
                fine_x_t = np.asarray(fine_x_t, dtype=np.float64)
                fine_y_t = np.asarray(fine_y_t, dtype=np.float64)
                fine_gt_t1 = fine_x_t[:, :3] + fine_y_t[:, :3]

                h_pred = x_t1[:, 0].detach().cpu().numpy()
                h_gt_fine = fine_gt_t1[:, 0]

                csi_fine = compute_grid_csi(
                    h_pred_nodes=h_pred,
                    pred_mesh=coarse_mesh,
                    h_gt_nodes=h_gt_fine,
                    fine_mesh=fine_mesh,
                    grid=grid,
                    threshold=threshold,
                    domain_mask=domain_mask,
                )
                agg[step]["csi_fine"].append(csi_fine)

            xn_t1 = _renorm(x_t1, mx, sx)
            if use_q_feature:
                q_t1_n = x_gt_full_n[:, 3:4]
                xn_t_full = torch.cat([xn_t1, q_t1_n], dim=1)
            else:
                xn_t_full = xn_t1

            g = g.clone()
            g.ndata["x"] = torch.cat([static_part, xn_t_full], dim=1)

    summary = {}
    for h in horizons_steps:
        values = np.asarray(agg[h]["csi_fine"], dtype=float)
        if values.size == 0:
            continue
        summary[h] = {
            "csi_fine_mean": float(np.nanmean(values)),
            "csi_fine_std": float(np.nanstd(values)),
        }
    return summary


def evaluate_single_checkpoint_fine_grid(
    ckpt_dir,
    epoch,
    horizons_steps,
    max_sequences=10,
    threshold=0.05,
    use_q_feature=USE_Q_FEATURE,
    data_dir=None,
    dynamic_dir=None,
    hydro_dir=None,
    fine_dynamic_dir=None,
    mesh=DEFAULT_MESH,
    coarse_mesh_path=None,
    fine_mesh_path=FINE_MESH_PATH,
    grid_resolution=GRID_RESOLUTION,
):
    resolved_data_dir = data_dir if data_dir is not None else DATA_DIRS[mesh]
    resolved_dynamic_dir = dynamic_dir if dynamic_dir is not None else DYNAMIC_DIR
    resolved_hydro_dir = hydro_dir if hydro_dir is not None else HYDRO_DIR
    resolved_fine_dynamic_dir = (
        fine_dynamic_dir if fine_dynamic_dir is not None else derive_fine_dynamic_files(resolved_dynamic_dir)
    )
    resolved_coarse_mesh_path = (
        coarse_mesh_path if coarse_mesh_path is not None else COARSE_MESH_PATHS[mesh]
    )

    sequence_length = max(horizons_steps) + 1
    ds = build_dataset(
        sequence_length=sequence_length,
        overlap=1,
        split="test",
        ckpt_dir=ckpt_dir,
        use_q_feature=use_q_feature,
        data_dir=resolved_data_dir,
        dynamic_dir=resolved_dynamic_dir,
        hydro_dir=resolved_hydro_dir,
    )
    fine_sequences = load_dynamic_sequences(
        resolved_fine_dynamic_dir,
        sequence_length=sequence_length,
        overlap=1,
    )

    expected_input_features = ds.base_graph.ndata["static"].shape[1] + (4 if use_q_feature else 3)
    model = build_model(expected_input_features)
    load_model_checkpoint(model, ckpt_dir, epoch=epoch)

    coarse_mesh = get_mesh(resolved_coarse_mesh_path)
    fine_mesh = get_mesh(fine_mesh_path)
    grid, domain_mask = get_common_grid(fine_mesh_path, grid_resolution)

    return evaluate_model_fine_grid(
        model=model,
        ds=ds,
        fine_sequences=fine_sequences,
        horizons_steps=horizons_steps,
        use_q_feature=use_q_feature,
        coarse_mesh=coarse_mesh,
        fine_mesh=fine_mesh,
        grid=grid,
        domain_mask=domain_mask,
        max_sequences=max_sequences,
        threshold=threshold,
    )


def evaluate_fixed_methods_fine_grid(methods, horizons_steps, max_sequences=10, threshold=0.05):
    fixed_metrics = {}
    for method in methods:
        name = method["name"]
        ckpt_dir = method["ckpt_dir"]
        use_q_feature = method.get("use_q_feature", USE_Q_FEATURE)
        dynamic_dir = method.get("dynamic_dir", DYNAMIC_DIR)
        hydro_dir = method.get("hydro_dir", HYDRO_DIR)
        fine_dynamic_dir = method.get("fine_dynamic_dir", None)
        mesh = method.get("mesh", DEFAULT_MESH)

        if "epochs" in method:
            epochs = list(method["epochs"])
        else:
            epochs = [method["epoch"]]

        for epoch in epochs:
            run_name = f"{name}@{epoch}"
            summary = evaluate_single_checkpoint_fine_grid(
                ckpt_dir=ckpt_dir,
                epoch=epoch,
                horizons_steps=horizons_steps,
                max_sequences=max_sequences,
                threshold=threshold,
                use_q_feature=use_q_feature,
                data_dir=method.get("data_dir", DATA_DIRS[mesh]),
                dynamic_dir=dynamic_dir,
                hydro_dir=hydro_dir,
                fine_dynamic_dir=fine_dynamic_dir,
                mesh=mesh,
                coarse_mesh_path=method.get("coarse_mesh_path", None),
                fine_mesh_path=method.get("fine_mesh_path", FINE_MESH_PATH),
                grid_resolution=method.get("grid_resolution", GRID_RESOLUTION),
            )
            fixed_metrics[run_name] = {
                "method": name,
                "mesh": mesh,
                "epoch": epoch,
                "ckpt_dir": ckpt_dir,
                "threshold_m": threshold,
                "summary": summary,
            }
    return fixed_metrics


def plot_hourly_csi_fine_grid(fixed_metrics, horizons_steps):
    if not fixed_metrics:
        print("Pas de resultats")
        return

    hours = np.array([step * DT_SECONDS / 3600.0 for step in horizons_steps], dtype=float)
    plt.figure(figsize=(12, 4))

    for run_name, payload in fixed_metrics.items():
        summary = payload["summary"]
        mean_vals = np.array(
            [summary.get(step, {}).get("csi_fine_mean", np.nan) for step in horizons_steps],
            dtype=float,
        )
        std_vals = np.array(
            [summary.get(step, {}).get("csi_fine_std", np.nan) for step in horizons_steps],
            dtype=float,
        )
        plt.plot(hours, mean_vals, marker="o", label=run_name)
        plt.fill_between(hours, mean_vals - std_vals, mean_vals + std_vals, alpha=0.18)

    plt.title(f"Fine-grid CSI vs horizon (threshold={THRESHOLD_M:.3f} m)")
    plt.xlabel("Horizon (hours)")
    plt.ylabel("Fine-grid CSI")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def fine_grid_metrics_to_dataframe(fixed_metrics):
    rows = []
    for run_name, payload in fixed_metrics.items():
        for h_step, vals in payload["summary"].items():
            rows.append(
                {
                    "run_name": run_name,
                    "method": payload["method"],
                    "mesh": payload["mesh"],
                    "epoch": payload["epoch"],
                    "horizon_steps": h_step,
                    "horizon_hours": h_step * DT_SECONDS / 3600.0,
                    "threshold_m": payload["threshold_m"],
                    "csi_fine_mean": vals.get("csi_fine_mean", np.nan),
                    "csi_fine_std": vals.get("csi_fine_std", np.nan),
                }
            )
    return pd.DataFrame(rows).sort_values(["method", "epoch", "horizon_steps"]).reset_index(drop=True)


In [ ]:
print(device)
print(f"Nombre de methodes: {len(METHODS)}")
print(f"HORIZONS_STEPS: {HORIZONS_STEPS}")
print(f"GRID_RESOLUTION: {GRID_RESOLUTION}")


In [ ]:
fixed_metrics_fine = evaluate_fixed_methods_fine_grid(
    METHODS,
    horizons_steps=HORIZONS_STEPS,
    max_sequences=MAX_SEQUENCES,
    threshold=THRESHOLD_M,
)
fixed_metrics_fine


In [ ]:
plot_hourly_csi_fine_grid(fixed_metrics_fine, HORIZONS_STEPS)


In [ ]:
df_fine = fine_grid_metrics_to_dataframe(fixed_metrics_fine)

if OUTPUT_CSV_PATH:
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_fine.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"CSV sauvegarde: {OUTPUT_CSV_PATH}")

df_fine
